In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


load_dotenv()

model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME"),  # pyright: ignore[reportArgumentType]
    temperature=0,
    base_url=os.environ.get("COMPATIBLE_BASE_URL"),
    api_key=os.environ.get("COMPATIBLE_API_KEY"),  # pyright: ignore[reportArgumentType]
)

embedding_model = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=os.environ.get("COMPATIBLE_API_KEY"),
)

client = QdrantClient(host="localhost", port=6333)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="agentic_rag_survey",
    embedding=embedding_model,
)

In [ ]:
from langchain.tools import tool


@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vectorstore.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs


tools = [retrieve_context]

In [3]:
SYSTEM_PROMPT = """You have access to a tool that retrieves context from a paper.
Use the tool to help answer user queries."""

In [4]:
from langchain.agents import create_agent


agent = create_agent(
    model=model,
    tools=[retrieve_context],
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
question = "What is RAG?"

for step in agent.stream(
    {"messages": question},  # pyright: ignore[reportArgumentType]
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is RAG?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_842ea1bd9b47460e9cdbab94)
 Call ID: call_842ea1bd9b47460e9cdbab94
  Args:
    query: What is RAG?
================================= Tool Message =================================
Name: retrieve_context

Source: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-05T01:26:00+00:00', 'author': '', 'keywords': '', 'moddate': '2025-02-05T01:26:00+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../example_data/A_SURVEY_ON_AGENTIC_RAG.pdf', 'total_pages': 39, 'page': 0, 'page_label': '1', 'start_index': 781, '_id': 'e984b0d2-381c-4368-81eb-9525475d40c6', '_collection_name': 'agentic_rag_survey'}
Content: outputs. Retri